# Lesson 1 — Gravitational Waves as Standard Sirens

**Cosmology Module (LVK Python Course)**

In this notebook we build a complete introduction to how gravitational-wave (GW) observations can be used as **standard sirens** to measure the expansion of the Universe.

---

## Learning goals

By the end of this lesson you should be able to:

1. Explain the difference between **bright sirens** and **dark sirens**.
2. Describe how the GW strain amplitude encodes the **luminosity distance** $d_L$.
3. Connect the distance-redshift relation $d_L(z)$ to measurements of the Hubble constant $H_0$.
4. Discuss the role of key events such as **GW170817** and population-level BBH dark-siren analyses.
5. Implement simple Python calculations and plots for low-redshift cosmology with sirens.


## Table of contents

1. Physical motivation
2. Standard candles vs standard sirens
3. Bright sirens and dark sirens
4. How GW data provide luminosity distance
5. The $d_L-z$ relation in cosmology
6. Measuring $H_0$ with sirens
7. Worked Python practice
8. Case study: GW170817 (bright siren)
9. Case study: BBH dark sirens
10. Practical limitations and systematics
11. Student exercises
12. References


---
## 1) Physical motivation

Modern cosmology relies on distance indicators. For nearby galaxies ($z\ll1$), Hubble's law links recession velocity and distance:

\[
cz \approx H_0 d.
\]

The challenge is to measure distances accurately and independently of traditional electromagnetic (EM) distance ladders.

Gravitational waves from compact binary coalescences provide an absolute distance scale because the waveform amplitude is predicted by General Relativity once source parameters are inferred. This makes them **standard sirens**.


---
## 2) Standard candles vs standard sirens

- **Standard candles** (e.g., Type Ia supernovae): infer distance from observed flux and assumed/calibrated intrinsic luminosity.
- **Standard sirens** (compact binary mergers): infer distance from waveform amplitude and phase evolution.

Why sirens are powerful:

- They do not require the cosmic distance ladder calibration chain.
- They are based on relativistic dynamics and wave propagation.
- They offer an independent cross-check on $H_0$ tension studies.


---
## 3) Bright and dark sirens

### Bright sirens

A GW event with an identified electromagnetic counterpart (e.g., kilonova/GRB/host galaxy).

- GW gives a posterior on $d_L$.
- EM counterpart gives a redshift $z$ (from host spectroscopy).
- Combine both to infer $H_0$.

**Prototype:** GW170817 + AT2017gfo + NGC 4993.

### Dark sirens

A GW event with no unique EM counterpart (typical for BBH mergers).

- GW gives sky map and distance posterior.
- Redshift is statistically assigned via all potential host galaxies in localization volume.
- Individual constraints are weak, but combining many events yields competitive cosmology.


---
## 4) How GW data provide luminosity distance

For a compact binary inspiral (schematically), the detector strain scales as

\[
h(t) \propto \frac{\mathcal{M}_z^{5/3} f(t)^{2/3}}{d_L} \times \text{(angular factors)}.
\]

Key points:

- Amplitude is inversely proportional to luminosity distance.
- The observed chirp mass is redshifted: $\mathcal{M}_z=(1+z)\mathcal{M}_{\rm source}$.
- Distance is partially degenerate with binary inclination and detector response.
- Network observations (multiple detectors) improve distance and sky localization.


In [ ]:
# Imports for the practical sections
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

c_km_s = 299792.458  # speed of light in km/s


---
## 5) The luminosity distance–redshift relation $d_L(z)$

For a flat $\Lambda$CDM cosmology,

\[
d_L(z)=(1+z)\frac{c}{H_0}\int_0^z\frac{dz'}{E(z')},
\qquad
E(z)=\sqrt{\Omega_m(1+z)^3+\Omega_\Lambda}.
\]

At low redshift ($z\lesssim 0.1$),

\[
d_L \approx \frac{cz}{H_0},
\]

which is the approximation frequently used for nearby sirens like GW170817.


In [ ]:
def dL_lcdm(z, H0=70.0, Om0=0.3, n_steps=4000):
    '''Luminosity distance in Mpc for flat LCDM using simple numerical integration.'''
    z = np.atleast_1d(z).astype(float)
    Ol0 = 1.0 - Om0
    out = np.zeros_like(z)

    for i, zi in enumerate(z):
        zz = np.linspace(0.0, zi, n_steps)
        Ez = np.sqrt(Om0 * (1 + zz)**3 + Ol0)
        integral = np.trapz(1.0 / Ez, zz)
        out[i] = (1 + zi) * (c_km_s / H0) * integral

    return out if out.size > 1 else out[0]


def dL_lowz(z, H0=70.0):
    '''Low-z approximation in Mpc.'''
    return (c_km_s / H0) * np.asarray(z)


In [ ]:
# Compare exact flat-LCDM dL(z) with low-z approximation
z = np.linspace(0.001, 0.2, 400)
H0_ref = 70.0

d_exact = dL_lcdm(z, H0=H0_ref, Om0=0.3)
d_lowz = dL_lowz(z, H0=H0_ref)
rel_err = (d_lowz - d_exact) / d_exact

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(z, d_exact, label=r"Flat $\Lambda$CDM")
ax[0].plot(z, d_lowz, "--", label=r"Low-$z$: $d_L\approx cz/H_0$")
ax[0].set_xlabel("Redshift z")
ax[0].set_ylabel("Luminosity distance [Mpc]")
ax[0].set_title("Distance-redshift relation")
ax[0].legend()

ax[1].plot(z, 100 * rel_err)
ax[1].axhline(0, color="k", lw=1)
ax[1].set_xlabel("Redshift z")
ax[1].set_ylabel("Relative error [%]")
ax[1].set_title("Low-z approximation error")

plt.tight_layout()
plt.show()


---
## 6) From $(d_L, z)$ to $H_0$

At low redshift,

\[
H_0 \approx \frac{cz}{d_L}.
\]

This gives intuition, but real analyses use full Bayesian inference with:

- detector-calibrated GW likelihood,
- peculiar velocity and redshift uncertainties,
- selection effects,
- and for dark sirens, galaxy catalog incompleteness.

Still, the simple estimator is useful for quick pedagogical experiments.


In [ ]:
# Simple Monte Carlo propagation for H0 from one bright siren
rng = np.random.default_rng(42)

# Toy values representative of a nearby event
z_mean, z_sigma = 0.0103, 0.0005
dL_mean, dL_sigma = 40.0, 7.0  # Mpc

n = 200_000
z_samp = rng.normal(z_mean, z_sigma, n)
dL_samp = rng.normal(dL_mean, dL_sigma, n)

mask = (z_samp > 0) & (dL_samp > 0)
H0_samp = c_km_s * z_samp[mask] / dL_samp[mask]

q16, q50, q84 = np.percentile(H0_samp, [16, 50, 84])
print(f"H0 median = {q50:.1f} km/s/Mpc")
print(f"68% interval = [{q16:.1f}, {q84:.1f}] km/s/Mpc")

plt.figure(figsize=(6.4, 4.2))
plt.hist(H0_samp, bins=120, density=True, alpha=0.8)
plt.axvline(q50, color="k", lw=2, label=f"median={q50:.1f}")
plt.axvspan(q16, q84, color="orange", alpha=0.25, label="68% interval")
plt.xlabel(r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]")
plt.ylabel("Posterior density (toy)")
plt.title("Toy bright-siren inference")
plt.legend()
plt.show()


---
## 7) Why dark sirens need populations

A single dark siren generally gives broad $H_0$ posteriors. Statistically,
combining many independent events sharpens constraints roughly like $\sigma(H_0)\propto N^{-1/2}$ in an idealized regime.

The next cell illustrates this scaling with synthetic posteriors.


In [ ]:
# Toy demonstration: combining N independent dark-siren-like Gaussian posteriors
H0_true = 70.0
single_sigma = 20.0
Ns = np.array([1, 2, 5, 10, 20, 50, 100])
combined_sigma = single_sigma / np.sqrt(Ns)

plt.figure(figsize=(6.4, 4.2))
plt.loglog(Ns, combined_sigma, "o-", label=r"$\sigma_N = \sigma_1/\sqrt{N}$")
plt.xlabel("Number of events N")
plt.ylabel(r"Expected 1$\sigma$ width on $H_0$ [km s$^{-1}$ Mpc$^{-1}$]")
plt.title("Idealized precision gain from event stacking")
plt.legend()
plt.show()


---
## 8) Case study: GW170817 (bright siren)

GW170817 was the first binary neutron star detection with a confirmed electromagnetic counterpart.

- **GW signal:** detected by LIGO Hanford, LIGO Livingston, and Virgo.
- **EM counterpart:** short GRB 170817A and kilonova AT2017gfo.
- **Host galaxy:** NGC 4993, providing a measured redshift.

Cosmological impact:

- First direct bright-siren estimate of $H_0$.
- Demonstrated feasibility of distance-ladder-independent cosmology.
- Highlighted major uncertainties: inclination-distance degeneracy and peculiar velocity corrections at low $z$.


---
## 9) BBH dark sirens and statistical cosmology

Most BBH mergers are dark sirens (no unique EM counterpart).

Method outline:

1. Build GW posterior over sky position and distance.
2. Query galaxy catalog in the localization volume.
3. Marginalize over candidate hosts and redshift uncertainties.
4. Include selection effects and catalog incompleteness.
5. Combine many events hierarchically.

Large BBH samples from GWTC catalogs are now routinely used to derive dark-siren constraints on $H_0$, typically broader than bright-siren constraints per event but increasingly informative in aggregate.


---
## 10) Practical limitations and systematics

When interpreting siren cosmology, keep track of:

- detector calibration uncertainties,
- waveform modeling systematics,
- weak lensing magnification (more relevant at high $z$),
- peculiar velocities (critical at low $z$),
- galaxy catalog incompleteness and selection biases,
- assumptions on source population priors.

Robust analyses propagate these consistently in hierarchical Bayesian frameworks.


---
## 11) Student exercises

1. **Low-z approximation test:** quantify up to what redshift the approximation $d_L\approx cz/H_0$ stays within 1%, 2%, and 5% of flat $\Lambda$CDM for $\Omega_m=0.3$.
2. **Single-event inference:** modify the toy bright-siren uncertainties and examine how the $H_0$ posterior width changes with distance uncertainty and redshift uncertainty.
3. **Selection bias thought experiment:** simulate a detection threshold in SNR and study how truncation in distance affects naive $H_0$ inference.
4. **Dark-siren stacking:** generate synthetic dark-siren posteriors with varying widths and verify when the $N^{-1/2}$ scaling breaks.
5. **GW170817 sensitivity:** explore how an assumed peculiar velocity correction (e.g., $\pm 250$ km/s) shifts low-z $H_0$ estimates.
6. **Extension topic (optional):** replace flat $\Lambda$CDM with a $w$CDM model and test parameter degeneracies between $H_0$ and $\Omega_m$ at low vs moderate redshift.


---
## 12) References

1. B. P. Abbott et al. (LIGO Scientific Collaboration and Virgo Collaboration), *A gravitational-wave standard siren measurement of the Hubble constant*, Nature **551**, 85–88 (2017).
2. B. P. Abbott et al. (LIGO Scientific Collaboration, Virgo Collaboration, et al.), *GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral*, Phys. Rev. Lett. **119**, 161101 (2017).
3. B. P. Abbott et al. (LIGO/Virgo), *A Gravitational-wave Measurement of the Hubble Constant Following the Second Observing Run of Advanced LIGO and Virgo*, ApJ **909**, 218 (2021).
4. R. Gray et al., *Cosmological inference using gravitational wave standard sirens: A mock data analysis*, Phys. Rev. D **101**, 122001 (2020).
5. S. Mukherjee, B. D. Wandelt, J. Silk, *Multimessenger tests of gravity and dark energy from GW standard sirens*, MNRAS **494**, 1956–1975 (2020).
6. LIGO-Virgo-KAGRA public catalogs (GWTC) and GWOSC documentation for event products and posterior samples.

---

### Suggested next lesson
Bayesian inference foundations with Bilby: priors, likelihoods, posteriors, and nested sampling for compact-binary parameter estimation.
